In [1]:
# GLOBAL #
import random
import pandas as pd
from collections import defaultdict, Counter
import numpy as np
from itertools import product
from scipy.stats import truncnorm
import matplotlib.pyplot as plt

random.seed(123)
np.random.seed(123)

In [2]:
# HELPER FUNCTIONS #

## From Markov Sampling ##

#create strings from the .csv file (for fish)
def readSequences(species_df, by_length=True):

        sequences = []

        grouped = species_df.groupby(
            ["trial_id", "sequence_id", "fish_id"]
        )

        for (_, _, fish_id), group in grouped:

            group = group.sort_values("order_id")
            parts = []

            for _, row in group.iterrows():

                if by_length:
                    parts.append(str(row["state"]) * int(row["length"]))
                else:
                    parts.append(str(row["state"]))

            sequences.append("".join(parts) + "")

        return sequences

# collapse sequence by self-transitions (e.g. [aaaabbbbcccdddccc] = [abcdc]) for collapsed context model
def collapseLength(seq):

    collapsed = []

    for s in seq:
        if not collapsed or collapsed[-1] != s:
            collapsed.append(s)

    return collapsed

# check if tuple has consecutive repeated states (we want to ignore these combinations in collapsed context model transition matrix)
def hasConsecutiveRepeats(states):

    for i in range(len(states) - 1):
        if states[i] == states[i + 1]:
            return True
        
    return False

# find indices of first state in each state run within sequence (e.g., [aaaabbbbcccdddccccc] = [0, 4, 8, 11, 14])
def uniqueIdxs(seq):

    lasts = []
    runChar = seq[0]

    for i, s in enumerate(seq[1:], start=1):
        if s != runChar:
            lasts.append(i - 1)
            runChar = s

    lasts.append(len(seq) - 1)
    return lasts

# expand  numeric vector into repetitions corresponding to each number value (e.g., [0, 4, 8, 11, 14] = [0, 0, 0, 0, 4, 4, 4, 4, 8, 8, 8, 11, 11, 11, 14, 14, 14]) 
# for matching strings to indices in collapsed context model 
def expandVec(vec):

    if not vec:
        return []
    
    result = []

    vecIDX = 0

    for pos in range(vec[-1] + 1):
        while vecIDX < len(vec) - 1 and pos > vec[vecIDX]:
            vecIDX += 1
        result.append(vec[vecIDX])

    return result

#### transition matrices ####

def collapsedTransNorm(states, lines, k, smoother):
    
    transitions = defaultdict(Counter)

    #for k=0 version, transition matrix turns into simple marginal state distributions
    if k == 0:
        for state in states:
            transitions[()][state] = smoother #add laplace smoother to each curr state

        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1 #add one count in corresponding transition cell for each instance of each state

        total = sum(transitions[()].values())

        return {(): {s: c / total for s, c in transitions[()].items()}} #normalize

    for prev in product(states, repeat=k):

        if hasConsecutiveRepeats(prev): #if prev tuple has consecutive repeats, skip (because this cannot be observed when self-transitions are collapsed)
            continue

        for curr in states:
            transitions[prev][curr] = smoother #add laplace smoother to every valid combination of prev & curr states

    for seq in lines:

        idxs = uniqueIdxs(seq) #find indices of unique states in sequence
        idxsExpanded = expandVec(idxs) #expand indices for mapping

        for t in range(1, len(seq)):

            idxsCollapsed = idxs.index(idxsExpanded[t - 1]) #find corresponding index of each token

            if idxsCollapsed < k - 1:
                continue

            prevIdxsUnique = idxs[idxsCollapsed- (k - 1):idxsCollapsed+ 1]
            prev = tuple(seq[i] for i in prevIdxsUnique)
            curr = seq[t]
            transitions[prev][curr] += 1

    transNorm = {}

    for cond, counter in transitions.items():
        total = sum(counter.values())
        transNorm[cond] = {s: c / total for s, c in counter.items()} #normalize 

    return transNorm

def collapsedTransCount(lines, k):

    transitions = defaultdict(Counter)

    # if k is zero, just count instances of each state
    if k == 0:
        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1

        return transitions
    
    for seq in lines:
        idxs = uniqueIdxs(seq)
        idxsExpanded = expandVec(idxs)

        for t in range(1, len(seq)):
            idxsCollapsed= idxs.index(idxsExpanded[t - 1])

            if idxsCollapsed< k - 1:
                continue

            prevIdxsUnique = idxs[idxsCollapsed- (k - 1):idxsCollapsed+ 1]
            prev = tuple(seq[i] for i in prevIdxsUnique)
            curr = seq[t]
            transitions[prev][curr] += 1

    return transitions

## Generation Functions ##

#initiate starting context based on count frequency
def sample_start_context(counts):

    contexts = list(counts.keys()) #all observed contexts from count transitions


    weights = np.array(
        [sum(nexts.values()) for nexts in counts.values()],
        dtype=float
    )

    #compute probs of each transition
    weights /= weights.sum()

    # select starting context according to its frequency
    idx = np.random.choice(len(contexts), p=weights)

    return contexts[idx]

# recover unique states from sequence
def collapsed_context(seq, k):

    if k == 0:
        return ()

    collapsed = collapseLength(seq)

    if len(collapsed) < k:
        return None

    return tuple(collapsed[-k:])

#sample next state from normalized matrix
def sample_next(trans, context):

    probs = trans[context]

    states = list(probs.keys())
    weights = list(probs.values())

    return np.random.choice(states, p=weights)

#generate sequence
def generate_sequence(trans, counts, k, length):

    start_context = sample_start_context(counts)

    seq = list(start_context)

    while len(seq) < length:

        context = collapsed_context(seq, k)

        if context not in trans:
            break

        probs = trans[context]

        states = list(probs.keys())
        weights = list(probs.values())

        next_state = np.random.choice(states, p=weights)

        seq.append(next_state)

    return ''.join(seq)


In [3]:
# FISH #

# user input -------

#note: "mil" is a typo for "mee" for "meeli"--I can't read apparently...
fishMap = {'bre': [1, .001], 'mil': [1, .1], 'mul': [2, .1], 'oce': [1, .001], 'orn': [1, .001], 'pul': [1, .1]} # dict for species: [best k, best smoother], extracted from 05-postprocessing results

nLines = 100

# run generation --------------

dat = pd.read_csv("../fish-data/raw/PrettyDat.csv")

for species, speciesDF in dat.groupby("species"):

    species = str(species)
    outFilename = f"{species}_playback.txt"

    #read all lines
    lines = readSequences(speciesDF)

    # find average sequence length (for generation length)
    avgLen = round(sum(len(line) for line in lines) / len(lines))

    # find unique states
    states = sorted({char for line in lines for char in line})

    # k-order model
    trans = collapsedTransNorm(states = states, lines = lines, k = fishMap[species][0], smoother = fishMap[species][1]) 

    # k-order counts
    counts = collapsedTransCount(lines, fishMap[species][0])

    # write generated sequences
    with open(outFilename, "w") as f:

        for i in range(nLines):

            out = generate_sequence(trans, counts, fishMap[species][0], avgLen)

            f.write(out + "\n")


In [ ]:
# BIRDS #

# user input -------

birdMap = {'bird1': [3, .1, 52, 117], 'bird2': [4, .1, 33, 54], 'bird3': [2, .1, 28, 64], 'bird4': [3, .1, 13, 73], 'bird5': [2, .1, 0, 50], 'bird6': [3, .1, 52, 129]} 
# dict for bird: [best k, best smoother, q1, q3], extracted from 05-postprocessing results and 01-preprocessing statsTable

nLines = 100

# run generation --------------

birds = ["bird1", "bird2", "bird3", "bird4", "bird5", "bird6"]

prefixes = pd.read_csv("../bird-data/prefixes.csv")

for bird in birds:

    filename = f"../bird-data/raw/{bird}_cropped_full.txt" 

    with open(filename) as f:
        lines = f.read().splitlines() 

    lines = [x for x in lines if x]

    # find average sequence length (for generation length)
    # avgLen = round(sum(len(line) for line in lines) / len(lines))

    # set up length distribution to draw from truncated normal distribution between q1 and q3 of each corpus
    q1 = birdMap[bird][2]
    q3 = birdMap[bird][3]

    mu = (q1 + q3) / 2
    sigma = (q3 - q1) / (2 * 0.67448975)

    #beginning and end of truncated normal are not simply q1 and q3, but mean-centered and renormalized versions (see scipy.stats documentation)
    a, b = (q1 - mu)/sigma, (q3 - mu)/sigma 

    # find unique states
    states = sorted({char for line in lines for char in line})

    # k-order model
    trans = collapsedTransNorm(states = states, lines = lines, k = birdMap[bird][0], smoother = birdMap[bird][1]) 

    # k-order counts
    counts = collapsedTransCount(lines, birdMap[bird][0])

    outFilename = f"{bird}_playback.txt"

    #write generated sequences
    with open(outFilename, "w") as f:

        for i in range(nLines):

            prefix = random.choice(
                prefixes.loc[prefixes["bird"] == bird, "fullPrefix"].tolist()
            )
            
            seqLen = truncnorm.rvs(a, b, loc=mu, scale=sigma, size=1) #sample sequence length from truncated N dist

            generated = generate_sequence(trans, counts, birdMap[bird][0], seqLen)

            out = prefix + generated

            f.write(out + "\n")


In [ ]:
# #print tables


# fishMap = {'bre': [1, .001], 'mil': [1, .1], 'mul': [2, .1], 'oce': [1, .001], 'orn': [1, .001], 'pul': [1, .1]} # dict for species: [best k, best smoother], extracted from 05-postprocessing results

# dat = pd.read_csv("../fish-data/raw/PrettyDat.csv")

# for species, speciesDF in dat.groupby("species"):

#     species = str(species)

#     lines = readSequences(speciesDF)
#     states = sorted({char for line in lines for char in line})

#     trans = collapsedTransNorm(
#         states=states,
#         lines=lines,
#         k=fishMap[species][0],
#         smoother=fishMap[species][1]
#     )

#     next_states = sorted({
#         s
#         for probs in trans.values()
#         for s in probs.keys()
#     })

#     print("\\begin{longtable}{l" + "l" * len(next_states) + "}")
#     print("\\toprule")
#     print("History & " + " & ".join(next_states) + r" \\")
#     print("\\midrule")
#     print("\\endfirsthead")

#     print("\\toprule")
#     print("History & " + " & ".join(next_states) + r" \\")
#     print("\\midrule")
#     print("\\endhead")

#     # Rows
#     for history in sorted(trans.keys()):

#         if isinstance(history, tuple):
#             hist = "".join(map(str, history))
#         else:
#             hist = str(history)

#         row = [hist]
#         for s in next_states:
#             val = trans[history].get(s, 0)

#             if val > fishMap[species][1]:
#                 row.append(f"\\cellcolor{{red!20}} {val:.3f}")
#             else:
#                 row.append(f"{val:.3f}")

#         print(" & ".join(row) + r" \\")

#     print("\\bottomrule")
#     print("\\end{longtable}")
#     print()


% ===== bre =====
\begin{longtable}{lllllllllll}
\caption{Transition matrix for bre}\\
\toprule
History & a & b & c & d & e & f & g & h & j & k \\
\midrule
\endfirsthead
\toprule
History & a & b & c & d & e & f & g & h & j & k \\
\midrule
\endhead
a & \cellcolor{red!20} 0.987 & 0.000 & \cellcolor{red!20} 0.001 & 0.000 & \cellcolor{red!20} 0.011 & 0.000 & 0.000 & 0.000 & 0.000 & 0.000 \\
b & 0.000 & \cellcolor{red!20} 0.996 & \cellcolor{red!20} 0.002 & 0.000 & 0.000 & 0.001 & \cellcolor{red!20} 0.001 & 0.000 & 0.000 & 0.000 \\
c & \cellcolor{red!20} 0.003 & \cellcolor{red!20} 0.002 & \cellcolor{red!20} 0.981 & \cellcolor{red!20} 0.002 & \cellcolor{red!20} 0.003 & \cellcolor{red!20} 0.001 & \cellcolor{red!20} 0.003 & \cellcolor{red!20} 0.003 & 0.000 & \cellcolor{red!20} 0.002 \\
d & 0.000 & 0.000 & \cellcolor{red!20} 0.004 & \cellcolor{red!20} 0.994 & 0.000 & 0.000 & 0.000 & 0.000 & \cellcolor{red!20} 0.002 & 0.000 \\
e & \cellcolor{red!20} 0.013 & 0.000 & \cellcolor{red!20} 0.001 & 0.0

In [ ]:
# birdMap = {'bird1': [3, .1], 'bird2': [4,.1], 'bird3': [2, .1], 'bird4': [3, .1], 'bird5': [2, .1], 'bird6': [3, .1]} # dict for bird: [best k, best smoother], extracted from 05-postprocessing results


# birds = ["bird1", "bird2", "bird3", "bird4", "bird5", "bird6"]


# for bird in birds:

#     filename = f"../bird-data/raw/{bird}_cropped_full.txt" 

#     with open(filename) as f:
#         lines = f.read().splitlines() 

#     lines = [x for x in lines if x]

#     states = sorted({char for line in lines for char in line})

#     trans = collapsedTransNorm(states = states, lines = lines, k = birdMap[bird][0], smoother = birdMap[bird][1]) 

#     next_states = sorted({
#         s
#         for probs in trans.values()
#         for s in probs.keys()
#     })

#     print("\\begin{longtable}{l" + "l" * len(next_states) + "}")
#     print("\\toprule")
#     print("History & " + " & ".join(next_states) + r" \\")
#     print("\\midrule")
#     print("\\endfirsthead")

#     print("\\toprule")
#     print("History & " + " & ".join(next_states) + r" \\")
#     print("\\midrule")
#     print("\\endhead")

#     # Rows
#     for history in sorted(trans.keys()):

#         if isinstance(history, tuple):
#             hist = "".join(map(str, history))
#         else:
#             hist = str(history)

#         row = [hist]
#         for s in next_states:
#             val = trans[history].get(s, 0)

#             if val > birdMap[bird][1]:
#                 row.append(f"\\cellcolor{{red!20}} {val:.3f}")
#             else:
#                 row.append(f"{val:.3f}")

#         print(" & ".join(row) + r" \\")

#     print("\\bottomrule")
#     print("\\end{longtable}")
#     print()



% ===== bird1 =====
\begin{longtable}{lllllllll}
\caption{Transition matrix for bird1}\\
\toprule
History & a & b & e & g & h & l & q & u \\
\midrule
\endfirsthead
\toprule
History & a & b & e & g & h & l & q & u \\
\midrule
\endhead
aba & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 \\
abe & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 \\
abg & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 \\
abh & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolor{red!20} 0.125 & \cellcolo